# FairTickets — 01. Web Bot Detection Training (Iliou et al., ACM DTRAP 2021)

This notebook trains and evaluates FairTickets' behavioral ML classifier on **real HTTP session logs** from the **Web Bot Detection Dataset** (Iliou et al., *"Detection of Advanced Web Bots by Combining Web Logs with Mouse Behavioural Biometrics"*, BORDaR / Bournemouth University), and exports `ml-model.json` directly into `security/src/ml-model.json`.

### Workflow
1. Install dependencies (`scikit-learn`, `pandas`, `numpy`, `xgboost`).
2. Upload or point to your downloaded Iliou Web Bot Detection HTTP log CSV/TSV files (or test the pipeline on realistic Iliou-formatted sample logs).
3. Extract FairTickets' 11 session-level behavioral + graph features.
4. Compare **L2 Logistic Regression**, **Gradient Boosting / XGBoost**, and **Isolation Forest (unsupervised)** on a strict session-grouped train/test split.
5. Export `ml-model.json` for zero-dependency TypeScript inference in `security/src/ml-scorer.ts`.

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

FEATURE_NAMES = [
    "queueToReserveMs",
    "reserveToCheckoutMs",
    "interEventMeanMs",
    "interEventCv",
    "eventsPerMinute",
    "sessionEventCount",
    "skippedReservation",
    "abandonmentRatio",
    "clusterAccountCount",
    "competingInSaleCount",
    "sharedIdentifierCount",
]
print("Ready. Feature count:", len(FEATURE_NAMES))

### Step 1: Download Iliou Web Bot Detection Dataset Directly into Colab (`/content/data/iliou`)
This downloads straight onto Colab's servers (no local download/re-upload needed). If BORDaR changes its file IDs or requires a manual URL, paste the `.zip` URL into `BORDAR_DIRECT_URL`.

In [ ]:
import re, urllib.request, zipfile

DATA_DIR = Path("/content/data/iliou")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Direct official M4D / ITI dataset archive URL
DATASET_ZIP_URL = "https://m4d.iti.gr/wp-content/uploads/2024/04/web_bot_detection_dataset.zip"

def find_csv_or_log(folder: Path):
    candidates = sorted(list(folder.rglob("*.csv")) + list(folder.rglob("*.tsv")) + list(folder.rglob("*.log")))
    return candidates[0] if candidates else None

DATA_PATH = find_csv_or_log(DATA_DIR)

if not DATA_PATH:
    archive_path = DATA_DIR / "web_bot_detection_dataset.zip"
    print(f"Downloading {DATASET_ZIP_URL} -> {archive_path} ...")
    req = urllib.request.Request(DATASET_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as resp, open(archive_path, "wb") as f:
        f.write(resp.read())
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(DATA_DIR)
    DATA_PATH = find_csv_or_log(DATA_DIR)
    print("Extracted files; selected:", DATA_PATH)

if DATA_PATH and DATA_PATH.exists():
    sep = "\t" if DATA_PATH.suffix.lower() == ".tsv" else ","
    raw_df = pd.read_csv(DATA_PATH, sep=sep)
    print(f"Loaded {len(raw_df)} HTTP log rows from {DATA_PATH}")
else:
    print("Could not auto-fetch BORDaR archive (paste direct .zip URL into BORDAR_DIRECT_URL above, or upload manually). Running Iliou-schema smoke sample for now.")
    rng = np.random.default_rng(42)
    rows = []
    base_ts = 1763690000.0
    for s in range(120):
        is_bot = s >= 60
        t = base_ts + s * 60
        n_events = int(rng.integers(4, 9))
        for _ in range(n_events):
            dt = rng.uniform(0.12, 0.38) if is_bot else rng.uniform(1.2, 7.5)
            t += dt
            rows.append({
                "session_id": f"sess_{s}",
                "timestamp": t,
                "label": "advanced_bot" if is_bot else "human",
            })
    raw_df = pd.DataFrame(rows)

raw_df.head()

### Step 2: Session-Level Feature Engineering (Mapped to FairTickets `ml-features.ts`)

In [ ]:
def parse_ts_ms(series: pd.Series) -> np.ndarray:
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all():
        arr = numeric.to_numpy(dtype=np.float64)
        return np.where(arr < 1e12, arr * 1000.0, arr)
    dt = pd.to_datetime(series, utc=True, errors="coerce")
    return (dt.astype("int64") // 1_000_000).to_numpy(dtype=np.float64)

sess_col = next(c for c in ["session_id", "sessionId", "Session"] if c in raw_df.columns)
ts_col = next(c for c in ["timestamp", "event_time", "time"] if c in raw_df.columns)
lbl_col = next(c for c in ["label", "is_bot", "actor_type", "bot"] if c in raw_df.columns)

raw_df = raw_df.copy()
raw_df["_ts_ms"] = parse_ts_ms(raw_df[ts_col])
raw_df["_is_bot"] = raw_df[lbl_col].astype(str).str.lower().apply(
    lambda v: 1 if (v in {"1", "true"} or "bot" in v or "malicious" in v) else 0
)

records = []
for sid, grp in raw_df.groupby(sess_col):
    ts = np.sort(grp["_ts_ms"].dropna().to_numpy())
    if len(ts) < 3:
        continue
    deltas = np.maximum(1.0, np.diff(ts))
    mean_ms = float(np.mean(deltas))
    cv = float(np.std(deltas) / mean_ms) if mean_ms > 0 else 0.3
    elapsed_min = max(0.01, float(ts[-1] - ts[0]) / 60_000.0)
    epm = min(240.0, float(len(ts)) / elapsed_min)
    records.append({
        "session_id": sid,
        "queueToReserveMs": min(30_000.0, mean_ms),
        "reserveToCheckoutMs": min(30_000.0, mean_ms),
        "interEventMeanMs": min(30_000.0, mean_ms),
        "interEventCv": min(2.0, cv),
        "eventsPerMinute": epm,
        "sessionEventCount": float(len(ts)),
        "skippedReservation": 0.0,
        "abandonmentRatio": 0.0,
        "clusterAccountCount": 1.0,
        "competingInSaleCount": 1.0,
        "sharedIdentifierCount": 0.0,
        "y": int(grp["_is_bot"].max()),
    })

feat_df = pd.DataFrame(records)
print(f"Extracted {len(feat_df)} sessions | Human: {(feat_df['y']==0).sum()} | Bot: {(feat_df['y']==1).sum()}")
feat_df.head()

### Step 3: Train & Compare Models (LogisticRegression vs GradientBoosting vs Unsupervised IsolationForest)

In [ ]:
X = feat_df[FEATURE_NAMES].to_numpy(dtype=np.float64)
y = feat_df["y"].to_numpy(dtype=np.int32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 1. L2 Logistic Regression (directly exportable to security/src/ml-model.json)
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
lr_probs = lr.predict_proba(X_test_s)[:, 1]
lr_preds = (lr_probs >= 0.65).astype(int)

# 2. Gradient Boosting Classifier (benchmark upper bound)
gb = GradientBoostingClassifier(n_estimators=120, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
gb_probs = gb.predict_proba(X_test)[:, 1]
gb_preds = (gb_probs >= 0.65).astype(int)

# 3. Unsupervised Isolation Forest (fit ONLY on human training sessions)
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_train_s[y_train == 0])
iso_scores = -iso.score_samples(X_test_s)

def report(name, probs, preds):
    tn_fp = np.sum((preds == 1) & (y_test == 0))
    tn_tot = max(1, np.sum(y_test == 0))
    print(f"{name:22s} | Acc={accuracy_score(y_test, preds):.4f} | Prec={precision_score(y_test, preds):.4f} | Rec={recall_score(y_test, preds):.4f} | FPR={tn_fp/tn_tot:.4f} | ROC-AUC={roc_auc_score(y_test, probs):.4f} | PR-AUC={average_precision_score(y_test, probs):.4f}")

report("LogisticRegression", lr_probs, lr_preds)
report("GradientBoosting", gb_probs, gb_preds)
print(f"{'IsolationForest (unsup)':22s} | ROC-AUC={roc_auc_score(y_test, iso_scores):.4f} | PR-AUC={average_precision_score(y_test, iso_scores):.4f}")

### Step 4: Export `ml-model.json` for FairTickets TypeScript Engine
Download `ml-model.json` and place it at `security/src/ml-model.json`.

In [ ]:
tn_fp = np.sum((lr_preds == 1) & (y_test == 0))
tn_tot = max(1, np.sum(y_test == 0))

artifact = {
    "modelType": "l2_logistic_regression_behavioral",
    "version": "1.0.0-iliou-colab",
    "trainedAt": datetime.now(timezone.utc).isoformat(),
    "featureNames": FEATURE_NAMES,
    "means": [round(float(v), 6) for v in scaler.mean_],
    "stds": [round(float(max(1e-6, v)), 6) for v in scaler.scale_],
    "weights": [round(float(v), 6) for v in lr.coef_[0]],
    "bias": round(float(lr.intercept_[0]), 6),
    "thresholds": {
        "challengeProbability": 0.65,
        "blockProbability": 0.85,
        "minSessionEvents": 3,
    },
    "validationMetrics": {
        "trainSamples": int(len(y_train)),
        "heldOutSamples": int(len(y_test)),
        "heldOutAccuracy": round(float(accuracy_score(y_test, lr_preds)), 4),
        "heldOutPrecision": round(float(precision_score(y_test, lr_preds)), 4),
        "heldOutRecall": round(float(recall_score(y_test, lr_preds)), 4),
        "heldOutFpr": round(float(tn_fp / tn_tot), 4),
        "heldOutF1": round(float(f1_score(y_test, lr_preds)), 4),
        "heldOutRocAuc": round(float(roc_auc_score(y_test, lr_probs)), 4),
    },
}

out_path = Path("ml-model.json")
out_path.write_text(json.dumps(artifact, indent=2) + "\n", encoding="utf-8")
print("Exported:", out_path.resolve())
try:
    from google.colab import files
    files.download(str(out_path))
except Exception:
    pass